In [1]:
#final version 
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import contractions
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/mohamed/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/mohamed/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/mohamed/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/mohamed/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /home/mohamed/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [15]:
#load dataset
df = pd.read_csv("data.csv")
print(df['status'].value_counts()) 

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64


In [16]:
#delete null form statment
df = df.dropna()

df.isnull().sum()

Unnamed: 0    0
statement     0
status        0
dtype: int64

In [17]:
#clean text 


    
import re
import contractions
from nltk.corpus import stopwords


def clean_data(text):
    text = str(text)

    # 0. normalize apostrophes
    text = text.replace("’", "'")

    # 1. lowercase
    text = text.lower()

    text = contractions.fix(text)

    # 3. replace names
    text = re.sub(r'@\w+', ' <name> ', text)

    # 4. replace numbers
    text = re.sub(r'\d+', ' <number> ', text)

    # 5. remove punctuation except ?, ! and '
    text = re.sub(r"[^\w\s?!]", "", text)

    return text

df["cleaned_statement"] = df["statement"].apply(clean_data)

'''

import nltk
from nltk.stem import WordNetLemmatizer

# download if not already done
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

import nltk
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'  # adjective
    elif tag.startswith('V'):
        return 'v'  # verb
    elif tag.startswith('N'):
        return 'n'  # noun
    elif tag.startswith('R'):
        return 'r'  # adverb
    else:
        return 'n'

def lemmatize_tokens(tokens):
    tagged = pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

from nltk.tokenize import word_tokenize

df['tokens'] = df['cleaned_statement'].apply(word_tokenize)
df['tokens'] = df['tokens'].apply(lemmatize_tokens)
df["cleaned_statement"] = df["tokens"].apply(lambda x: " ".join(x))'''

'\n\nimport nltk\nfrom nltk.stem import WordNetLemmatizer\n\n# download if not already done\nnltk.download(\'punkt\')\nnltk.download(\'wordnet\')\nnltk.download(\'omw-1.4\')\n\nimport nltk\nfrom nltk import pos_tag\nfrom nltk.stem import WordNetLemmatizer\n\nlemmatizer = WordNetLemmatizer()\n\ndef get_wordnet_pos(tag):\n    if tag.startswith(\'J\'):\n        return \'a\'  # adjective\n    elif tag.startswith(\'V\'):\n        return \'v\'  # verb\n    elif tag.startswith(\'N\'):\n        return \'n\'  # noun\n    elif tag.startswith(\'R\'):\n        return \'r\'  # adverb\n    else:\n        return \'n\'\n\ndef lemmatize_tokens(tokens):\n    tagged = pos_tag(tokens)\n    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]\n\nfrom nltk.tokenize import word_tokenize\n\ndf[\'tokens\'] = df[\'cleaned_statement\'].apply(word_tokenize)\ndf[\'tokens\'] = df[\'tokens\'].apply(lemmatize_tokens)\ndf["cleaned_statement"] = df["tokens"].apply(lambda x: " ".join(x))'

In [18]:
from sklearn.preprocessing import LabelEncoder


# Encode labels
label_encoder = LabelEncoder()

df["status"] = label_encoder.fit_transform(df["status"])

In [19]:
#test data 
df["status"].value_counts()

status
3    16343
2    15404
6    10652
0     3841
1     2777
5     2587
4     1077
Name: count, dtype: int64

In [20]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["status"]   # keeps class balance
)

In [44]:
y_train.isna().sum()

np.int64(0)

In [21]:
from textblob import TextBlob
import pandas as pd

# Augmentation function
def augment_text(text):
    try:
        if not isinstance(text, str) or not text.strip():
            return text

        translated = (
            TextBlob(text)
            .translate(to='fr')
            .translate(to='en')
        )

        return str(translated)

    except Exception:
        return text


# Create augmented copy
augmented_df = df_train.copy()

# Apply augmentation
augmented_df["cleaned_statement"] = augmented_df[
    "cleaned_statement"
].apply(augment_text)

# Combine original + augmented
df_train = pd.concat([df_train, augmented_df], ignore_index=True)

# Shuffle
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)



In [23]:
# Reapply preprocessing on augmented data
df_train['cleaned_statement'] = df_train["cleaned_statement"].apply(clean_data)




In [180]:
X_train.apply(clean_data)

27354                          muzocan talk australia issue
36535     feel beyond awful right got hang little piece ...
6186      convinced take annoyed judge working nothing p...
93051     hi first want apologize mistakes make since en...
20535               thinking hope go bed tonight never wake
                                ...                        
54886     blew hella money joannâs ð damn hate love make...
76820     amber know often check really miss right like ...
103694    force talk people ? two weeks ago would sworn ...
860       pills need wrist bleeding bad think ever get b...
15795     guess need vent timid shy person social anxiet...
Name: cleaned_statement, Length: 84289, dtype: object

In [178]:
X_train.isna().sum()

np.int64(0)

In [83]:
#comparaision

In [24]:
#vectorisation

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=10000)

X_train_tfidf = vectorizer.fit_transform(df_train['cleaned_statement'])
X_test_tfidf = vectorizer.transform(df_test['cleaned_statement'])

In [25]:
#TEST
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

model = LogisticRegression(
    class_weight='balanced',max_iter=1000  
)
model.fit(X_train_tfidf,df_train["status"])


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import train_test_split, GridSearchCV

df = pd.read_csv("data.csv")


#clean text 
def clean_data(text):
    text = str(text)

    # 0. normalize apostrophes
    text = text.replace("’", "'")

    # 1. lowercase
    text = text.lower()

    text = contractions.fix(text)

    # 3. replace names
    text = re.sub(r'@\w+', ' <name> ', text)

    # 4. replace numbers
    text = re.sub(r'\d+', ' <number> ', text)

    # 5. remove punctuation except ?, ! and '
    text = re.sub(r"[^\w\s?!]", "", text)

    return text

df["cleaned_statement"] = df["statement"].apply(clean_data)


#encodage 
from sklearn.preprocessing import LabelEncoder


# Encode labels
label_encoder = LabelEncoder()

df["status"] = label_encoder.fit_transform(df["status"])

#seaprer dataset
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["status"]   # keeps class balance
)



#text augmetation 
from textblob import TextBlob
import pandas as pd

# Augmentation function
def augment_text(text):
    try:
        if not isinstance(text, str) or not text.strip():
            return text

        translated = (
            TextBlob(text)
            .translate(to='fr')
            .translate(to='en')
        )

        return str(translated)

    except Exception:
        return text


# Create augmented copy
augmented_df = df_train.copy()

# Apply augmentation
augmented_df["cleaned_statement"] = augmented_df[
    "cleaned_statement"
].apply(augment_text)

# Combine original + augmented
df_train = pd.concat([df_train, augmented_df], ignore_index=True)

# Shuffle
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)


# Reapply preprocessing on augmented data
df_train['cleaned_statement'] = df_train["cleaned_statement"].apply(clean_data)

#vectorisation

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=10000)

X_train_tfidf = vectorizer.fit_transform(df_train['cleaned_statement'])
X_test_tfidf = vectorizer.transform(df_test['cleaned_statement'])



# Model Training with Hyperparameter Tuning
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

model = LogisticRegression(max_iter=1000)
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, df_train["status"])

# Best Model
best_model = grid_search.best_estimator_

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_pred = best_model.predict(X_test_tfidf)
# Evaluation


print("Accuracy Score:")
print(accuracy_score(df_test["status"], y_pred))

print("Classification Report:")
print(classification_report(df_test["status"], y_pred))




In [142]:
scores

array([0.76075152, 0.76061337, 0.76748839, 0.76352766, 0.7594468 ])

In [208]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

y_pred = cross_val_predict(best_model, X_test_tfidf, y_test, cv=cv)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.75      0.77      1567
           1       0.85      0.70      0.77      1077
           2       0.70      0.73      0.71      6206
           3       0.86      0.92      0.89      6572
           4       0.80      0.55      0.65       381
           5       0.67      0.52      0.59      1080
           6       0.64      0.63      0.64      4190

    accuracy                           0.76     21073
   macro avg       0.76      0.69      0.72     21073
weighted avg       0.75      0.76      0.75     21073



In [29]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_pred = best_model.predict(X_test_tfidf)
# Evaluation


print("Accuracy Score:")
print(accuracy_score(df_test["status"], y_pred))

print("Classification Report:")
print(classification_report(df_test["status"], y_pred))



Accuracy Score:
0.7171870551390339
Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.73      0.76       768
           1       0.77      0.70      0.73       556
           2       0.65      0.66      0.65      3081
           3       0.88      0.90      0.89      3269
           4       0.75      0.61      0.68       215
           5       0.52      0.48      0.50       517
           6       0.57      0.58      0.58      2131

    accuracy                           0.72     10537
   macro avg       0.70      0.67      0.68     10537
weighted avg       0.72      0.72      0.72     10537

